# 08 · 端到端實戰:訓練 agent 玩接水果

整條軌道的收尾。我們把前面所有東西串起來:拿第 07 課自訂的 **CatchEnv**,用 **stable-baselines3** 訓練一個真的會接水果的 agent,評估它、存檔,最後聊聊**怎麼把訓練好的模型搬進瀏覽器**——這正是本站遊戲區 RL 的做法。

## 1. 安裝 + 載入環境

In [ ]:
%pip install -q -U "stable-baselines3>=2.0" "gymnasium[classic-control]"

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces


class CatchEnv(gym.Env):
    """接水果：底部木板接住掉落的水果。這是我們親手刻的 gymnasium 環境。

    觀察 observation：[paddle_x, fruit_x, fruit_y]，三者都正規化到 0~1。
    動作 action：0=左移、1=不動、2=右移。
    獎勵 reward：接到 +1、漏接 -1，其餘步為 0。
    回合 episode：不自然結束，靠 TimeLimit 包裝器設定步數上限。
    """

    metadata = {"render_modes": ["ansi"]}

    def __init__(self, grid_size=5, render_mode=None):
        super().__init__()
        self.grid = grid_size
        self.render_mode = render_mode
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(3,), dtype=np.float32
        )

    def _obs(self):
        n = self.grid - 1
        return np.array(
            [self.paddle / n, self.fruit_x / n, self.fruit_y / n], dtype=np.float32
        )

    def _spawn_fruit(self):
        self.fruit_x = int(self.np_random.integers(0, self.grid))
        self.fruit_y = self.grid - 1

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.paddle = self.grid // 2
        self._spawn_fruit()
        return self._obs(), {}

    def step(self, action):
        # 木板依動作左右移（action-1 → -1/0/+1），夾在邊界內。
        self.paddle = int(np.clip(self.paddle + (action - 1), 0, self.grid - 1))
        self.fruit_y -= 1
        reward = 0.0
        if self.fruit_y <= 0:                       # 水果落到底了，結算這顆
            reward = 1.0 if self.paddle == self.fruit_x else -1.0
            self._spawn_fruit()                     # 立刻補一顆新的，遊戲繼續
        return self._obs(), reward, False, False, {}

    def render(self):
        rows = []
        for y in range(self.grid - 1, -1, -1):
            row = ["."] * self.grid
            if y == self.fruit_y:
                row[self.fruit_x] = "*"
            if y == 0:
                row[self.paddle] = "U"
            rows.append(" ".join(row))
        return "\n".join(rows)

## 2. 包上 TimeLimit,組成訓練環境

CatchEnv 不會自然結束,用 `TimeLimit` 給每回合 200 步上限,方便統計與訓練。

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium.wrappers import TimeLimit
from stable_baselines3 import PPO


def make_env():
    return TimeLimit(CatchEnv(grid_size=5), max_episode_steps=200)


# 訓練前先量一個「隨機策略」基準,等下好對照
def avg_reward(model=None, episodes=30):
    env = make_env()
    scores = []
    for _ in range(episodes):
        obs, _ = env.reset()
        done, total = False, 0.0
        while not done:
            if model is None:
                action = env.action_space.sample()
            else:
                action, _ = model.predict(obs, deterministic=True)
            obs, r, term, trunc, _ = env.step(int(action))
            done = term or trunc; total += r
        scores.append(total)
    return float(np.mean(scores))


baseline = avg_reward(None)
print(f"隨機策略基準:每 200 步淨得分 {baseline:.1f}")

## 3. 訓練 PPO

約幾分鐘(T4 更快)。`total_timesteps` 拉高會更強。

In [ ]:
model = PPO("MlpPolicy", make_env(), verbose=0)
model.learn(total_timesteps=100_000)
print("訓練完成。")

## 4. 評估:跟基準比，進步多少

In [ ]:
trained = avg_reward(model)
print(f"隨機策略 :{baseline:6.1f}")
print(f"訓練後   :{trained:6.1f}   ← 接到水果越多分越高")

## 5. 看 agent 實際玩一回合

`U` 木板會主動移到 `*` 水果正下方去接。

In [ ]:
env = make_env()
obs, _ = env.reset(seed=1)
for step in range(8):
    action, _ = model.predict(obs, deterministic=True)
    obs, r, term, trunc, _ = env.step(int(action))
    print(env.render())
    print(f"step {step}  action={int(action)}  reward={r}")
    print("-" * 12)

## 6. 存檔與載入

訓練好的模型可存成檔案,日後直接載入推論,不用重訓。

In [ ]:
model.save("catch_ppo")
loaded = PPO.load("catch_ppo")
print("已存檔並重新載入:", loaded)

## 7. 從 Colab 到瀏覽器:本站怎麼做

你訓練的 agent 現在活在 Python 裡。本站的**遊戲區**把同樣的流程搬上線,讓 RL agent 直接在你的瀏覽器裡玩遊戲。關鍵幾招:

- **遊戲邏輯只寫一份(TypeScript)**:遊戲的 `GameCore` 用 TypeScript 寫,瀏覽器直接用;Python 訓練時則用 **PyMiniRacer(V8)** 跑同一份 JS,確保訓練與線上行為 100% 一致——遊戲規則不會有兩套。
- **Gymnasium 包裝**:Python 端寫一個 `gym.Env` 把那份 JS 的 `reset/step` 包起來(就像你這課對 CatchEnv 做的),交給 stable-baselines3 訓練。
- **匯出權重給瀏覽器**:訓練好的策略網路權重匯出成 JSON / TF.js 格式,前端載入後即可在瀏覽器即時推論。

> 想看真實版本,翻 repo 的 `ml-training/` 與遊戲區的 `GameCore` 架構。你這條軌道學的每一塊——環境介面、PPO 訓練、存檔——都是那套上線流程的縮影。

## 軌道小結

你從**完全不懂 RL**,一路手刻到能訓練 agent 玩自製遊戲:

1. RL 世界觀與 gymnasium 介面
2. 手刻 Q-learning(表格法)
3. 手刻 DQN(神經網路逼近)
4. stable-baselines3(站在巨人肩上)
5. 手刻策略梯度 REINFORCE
6. 獎勵塑形與環境包裝器
7. 自訂 gymnasium 環境
8. 端到端訓練 + 上線思路

**原理你手刻過、工具你也會用**——這正是把 RL 用在真實問題上需要的兩種底氣。🎮